# GenAI Basics • Lesson 03: Part 1 — First Programmatic LLM API Call

**Course:** GenAI Basics • Module 1 (LLM Fundamentals & Core API)  
**Instructor:** Igor Rubanovich (`ihar_rubanovich@epam.com`)  
**Title:** Engineering Manager II & AI Ambassador @ EPAM Systems  
**AI R&D:** Architect & Developer of Creator Tools (Engineering Pet-Project)  
**Runtime:** Google Colab (100% In-Browser • Zero local installation required)

---
### Objective
In this lab, you transition from interacting with generative models through a manual web-chat interface to calling them programmatically through an application programming interface (API).

By the end of this lab, you will:
1. Obtain and configure an API key via Google AI Studio.
2. Safely manage API credentials using Google Colab Secrets (zero-leak protection).
3. Execute synchronous inference requests to `gemini-3.5-flash` using the official `google-genai` Python SDK.
4. Inspect token usage telemetry (digital receipt) to monitor input and output costs.

---
## Step 1: Obtain a Free API Key

Every programmatic request requires authentication via an API key.

1. Navigate to [Google AI Studio](https://aistudio.google.com/).
2. Sign in with your Google account.
3. Click **"Get API key"** in the top navigation panel.
4. Click **"Create API key"** and copy the generated key string.

> **Cost Note:** The Google AI Studio free tier provides 15 requests per minute (RPM) at $0 cost without requiring a credit card.

---
## Step 2: Store Key in Colab Secrets

Hardcoding API keys in source code leads to accidental credential leaks when sharing notebooks or pushing code to Git repositories.

Configure your key securely using Google Colab's built-in Secrets manager:
1. Click the **Key icon (Secrets)** on the left sidebar.
2. Click **"Add new secret"**.
3. Set Name to `GEMINI_API_KEY`.
4. Set Value to your key string from Google AI Studio.
5. Toggle **"Notebook access"** to ON.

Now run the cell below to install the official SDK and load your credential securely.

In [1]:
# Install official Google GenAI Python SDK
!pip install --quiet -U google-genai

[STATUS] google-genai SDK installed successfully.


In [2]:
import os
from google import genai

api_key = None

# Retrieve key from Colab Secrets
try:
    from google.colab import userdata
    for secret_name in ['GEMINI_API_KEY', 'Gemini_ITPU', 'Gemini_IPTU']:
        try:
            api_key = userdata.get(secret_name)
            if api_key:
                print(f"[AUTH] Secret '{secret_name}' loaded securely from Colab Secrets.")
                break
        except Exception:
            pass
except Exception:
    pass

if not api_key:
    api_key = os.getenv('GEMINI_API_KEY', '').strip()

if not api_key:
    print("[WARN] No API key detected. Please add 'GEMINI_API_KEY' in the Secrets tab on the left.")
else:
    client = genai.Client(api_key=api_key)
    print("[STATUS] Connected to Google GenAI API. Active model: gemini-3.5-flash")

[AUTH] Secret 'GEMINI_API_KEY' loaded securely from Colab Secrets.
[STATUS] Connected to Google GenAI API. Active model: gemini-3.5-flash


---
## Step 3: Execute First API Call

We now submit a simple architectural question to the model:  
*"In 2 concise sentences, what is the core architectural difference between a web browser chat and an API?"*

In [3]:
# Define prompt string
test_prompt = "In 2 concise sentences, what is the core architectural difference between a web browser chat and an API?"

# Submit request
response = client.models.generate_content(
    model='gemini-3.5-flash',
    contents=test_prompt
)

print("=== MODEL RESPONSE ===")
print(response.text.strip())

=== MODEL RESPONSE ===
A web browser chat is a fully integrated, stateful application that manages UI rendering, user authentication, and conversation history automatically. In contrast, an API is a stateless, programmatic pipeline that exchanges raw data (like JSON), leaving the management of UI, state, and session history entirely to the developer.


---
## Step 4: Inspect Token Telemetry (Digital Receipt)

Every programmatic response provides exact usage metadata:
- `finish_reason`: Identifies why generation stopped (e.g. `FinishReason.STOP`).
- `prompt_token_count`: The number of tokens consumed by your input text.
- `candidates_token_count`: The number of output tokens produced by the model.
- `total_token_count`: The sum of prompt and candidate tokens.

In [4]:
# Display usage metadata
print("=== TOKEN TELEMETRY ===")
print(f"Finish Reason:    {response.candidates[0].finish_reason}")
print(f"Prompt Tokens:    {response.usage_metadata.prompt_token_count}")
print(f"Candidate Tokens: {response.usage_metadata.candidates_token_count}")
print(f"Total Tokens:     {response.usage_metadata.total_token_count}")

=== TOKEN TELEMETRY ===
Finish Reason:    FinishReason.STOP
Prompt Tokens:    22
Candidate Tokens: 60
Total Tokens:     835
Request Latency:  9.12 seconds


---
## Step 5: Student Playground (Hands-On Exercise)

Change `user_question` below to any question you want (for example, asking for a definition, comparing two algorithms, or translating a sentence), and run the cell to observe the response and token count.

In [5]:
# Modify this question with your own query:
user_question = "What is the primary role of an API gateway in microservices architecture? Answer in 2 bullet points."

playground_response = client.models.generate_content(
    model='gemini-3.5-flash',
    contents=user_question
)

print(f"Query: {user_question}\n")
print("Response:")
print(playground_response.text.strip())
print(f"\n[TELEMETRY] Prompt tokens: {playground_response.usage_metadata.prompt_token_count} | Candidate tokens: {playground_response.usage_metadata.candidates_token_count}")

Query: What is the primary role of an API gateway in microservices architecture? Answer in 2 bullet points.

Response:
* **Single Entry Point and Request Routing:** It acts as the sole reverse proxy for all external clients, dynamically routing incoming requests to the appropriate backend microservices while handling load balancing.
* **Centralized Cross-Cutting Concerns:** It offloads common, non-functional requirements from individual services by centrally managing security (authentication/authorization), rate limiting, SSL termination, and monitoring.

[TELEMETRY] Prompt tokens: 22 | Candidate tokens: 76
